# STEP12. Eye CNN 학습 · 조건 비교

## 분석 질문

- MRL 단독, DMD 단독, MRL+DMD 합본, MRL→DMD 미세조정 조건 중 어느 학습 방식이 DMD hold-out subject에서 Closed-Recall과 F1 측면에서 더 안정적인가.
- 특히 MRL+DMD 합본이 MRL 단독보다 Closed-Recall을 높이는지 비교한다.

## 비교 조건

| | 학습 소스 | 측정하는 것 |
|---|---|---|
| A | MRL only | 도메인 갭의 하한선 |
| B | DMD only | in-domain 상한선 (DMD 표본 10,956장으로 충분한가) |
| C | MRL + DMD | 대량 IR + 소량 RGB 결합 이득 |
| C' | MRL 사전학습 → DMD 미세조정 | 순차 적응이 합본보다 나은가 |

test 는 네 조건 모두 **DMD hold-out 4명 고정**이다. 그래야 비교가 성립한다.

## 입력

| 구분 | 경로 |
|---|---|
| 통합 manifest (STEP11) | `config.OUTPUTS_DIR / "eye_dataset" / "eye_manifest.csv"` |
| MRL · DMD 이미지 | manifest 의 저장소 루트 기준 상대경로 |

## 출력

| 파일 | 위치 |
|---|---|
| `eye_<tag>.keras` (조건별) | `config.ARTIFACT_DIR` |
| `eye_<tag>_metrics.json` | `config.ARTIFACT_DIR` |
| `step12_comparison.csv` | `config.OUTPUTS_DIR / "eye_dataset"` |

## 전체 수행 흐름

**PART A — 파이프라인 점검 (저비용)**

1. 설정·경로
2. 입력 텐서 규격 확인
3. 스모크 학습 (소규모 · 1 epoch)

**PART B — 본 실험 (128 gray)**

4. A) MRL only
5. B) DMD only
6. C) MRL + DMD
7. C') MRL → DMD 미세조정
8. 조건 비교

**PART C — 부록 · 연결 점검**

9. 부록: 256 RGB 조건
10. STEP03(YuNet 추론)에 연결 가능한지 규격 점검

## PART A — 파이프라인 점검

### 목적

- 전체 학습을 돌리기 전에 입력 규격과 학습·평가 경로에 오류가 없는지 확인한다.
- MRL 이 84,898장이라 전체 학습은 비싸다. 오류는 소규모에서 먼저 잡는다.

In [6]:
# [셀 1] 설정 · 경로 (import·경로·시드는 여기서 한 번만)

# --- 저장소 루트 부트스트랩 (모든 노트북 공통, 수정 금지) ---
import sys
from pathlib import Path

_anchors = []
if "__vsc_ipynb_file__" in globals():          # VS Code Notebook
    _anchors.append(Path(globals()["__vsc_ipynb_file__"]).resolve().parent)
if globals().get("_dh"):                        # IPython 커널 시작 폴더
    _anchors.append(Path(globals()["_dh"][0]).resolve())
_anchors.append(Path.cwd().resolve())           # 최후 수단

_root = next(
    (p for a in _anchors for p in [a, *a.parents]
     if (p / "config.py").is_file() and (p / "requirements.txt").is_file()),
    None,
)
if _root is not None:
    if str(_root) in sys.path:
        sys.path.remove(str(_root))
    sys.path.insert(0, str(_root))

import config

if _root is not None and Path(config.__file__).resolve().parent != _root:
    raise ImportError(f"의도하지 않은 config.py 가 import 되었습니다: {config.__file__}")
# --- 부트스트랩 끝 ---

_src = str(config.PROJECT_ROOT / "src")
if _src not in sys.path:
    sys.path.insert(0, _src)

import pandas as pd
import numpy as np
import train_eye as T

EYE_DS_DIR = config.OUTPUTS_DIR / "eye_dataset"
COMPARISON_CSV = EYE_DS_DIR / "step12_comparison.csv"

SIZE, GRAY = 128, True          # 본 실험 규격
EPOCHS = 20
RESULTS = {}                     # 조건별 결과. 셀을 따로 돌려도 누적된다.

df = T.load_manifest()
print("통합 manifest :", config._rel(T.MANIFEST), f"({len(df):,} rows)")
print("artifacts     :", config._rel(T.ARTIFACT_DIR))
print()
print(df.groupby(["split", "source"])
        .agg(n=("path", "size"),
             closed_pct=("class_idx", lambda s: round((s == 0).mean() * 100, 1)))
        .reset_index().to_string(index=False))

통합 manifest : outputs\eye_dataset\eye_manifest.csv (106,802 rows)
artifacts     : model\artifacts

split source     n  closed_pct
 test    dmd  7610        11.3
 test    mrl 12155        50.6
train    dmd 10956        51.4
train    mrl 60264        49.2
  val    dmd  3338        44.6
  val    mrl 12479        49.1


### 결정 박스 6 — 판정 임계값 선택

- 문제: Closed / Open 판정 임계값을 (a) 0.5 고정, (b) test에서 최적화, (c) val에서 선택 후 test에 적용하는 방법 중 무엇을 사용할 것인가.
- 선택: **(c). validation set에서 Closed-Recall ≥ 0.90을 만족하는 가장 높은 threshold를 선택하고, 이를 test에 고정 적용한다.**
- 근거: test에서 threshold를 최적화하면 test 정보가 모델 선택에 사용되어 평가가 낙관적으로 편향될 수 있다. 또한 클래스 불균형이 있는 test에서 0.5를 고정하면 Closed-Recall이 낮아질 수 있다.
- 본 분석에서는 Recall ≥ 0.90이라는 사전 제약 아래, validation set에서 선택 가능한 threshold 중 가장 높은 값을 선택한다.
- 참고용으로 threshold = 0.50 결과도 함께 확인하여 threshold 조정에 따른 변화를 분리한다.

### 결정 박스 7 — 평가 단위

- 문제: DMD는 프레임당 좌·우 눈 crop 2장이다. crop 단위로 평가할지, 프레임 단위로 통합할지 결정해야 한다.
- 선택: **프레임 단위. 좌·우 눈의 Closed 확률 중 max를 해당 프레임의 확률로 사용한다.**
- 근거: 동일 프레임에서 추출한 좌·우 눈 crop은 독립적인 표본으로 보기 어렵다. crop 단위로 평가하면 동일 프레임의 정보가 두 번 반영된다.
- max 규칙은 두 눈 중 한쪽이라도 감긴 경우 Closed로 판단하는 방식이며, STEP03의 판정 규칙과 동일한 기준을 사용한다.
- MRL은 `side = "-"`이므로 해당 값은 그대로 유지된다.
- 사전 계획.


### 결정 박스 8 — 입력 규격

- 문제: (a) 128×128 그레이스케일, (b) 256×256 RGB(STEP02 의 `eye_model.keras` 와 같은 규격).
- 선택: **본 실험 (a), 부록 (b).**
- 근거: MRL 은 IR 그레이스케일이라 색 정보가 없다. 3채널로 쓰려면 그레이를 복제해야 하고, 그러면 DMD 만 색 정보를 갖게 되어 모델이 "색이 있으면 DMD"를 학습할 여지가 생긴다. 채널을 1로 통일하는 것은 용량 절약이 아니라 두 데이터셋을 같은 입력 공간으로 보내는 조치다. 원본 눈 crop 이 85~300px 이라 256 은 업샘플링이기도 하다.
- (b)는 버리지 않는다. STEP03 에 그대로 로드할 수 있다는 실용적 가치가 있어 부록 1건으로 남긴다.


### 결정 박스 9 — 모델 선택 기준

- 문제: 주 평가 지표는 Closed-Recall인데 `ModelCheckpoint`는 `val_accuracy`를 기준으로 한다.
- 선택: **본 실험에서는 `val_accuracy`로 checkpoint를 선택하고, 최종 Closed-Recall은 validation에서 threshold를 조정하여 평가한다.**
- 근거: 현재 validation set의 클래스 비율이 MRL 49.1%, DMD 44.6%로 극단적으로 불균형하지 않으며, 단일 지표에 의존하지 않고 Recall·Precision·F1을 함께 확인한다.
- 한계: 모델 checkpoint 선택 자체가 Closed-Recall을 직접 최적화하는 것은 아니다. 따라서 본 결과를 Closed-Recall 기준의 최적 모델이라고 해석하지 않는다.
- 필요할 경우 후속 분석에서 validation Closed-Recall 기반 checkpoint 선택과 결과를 비교할 수 있다.
- 사전 계획.

### 목적

- 모델에 들어가는 텐서의 크기·채널·값 범위가 의도와 같은지 한 배치로 확인한다.

In [2]:
# [셀 2] 입력 텐서 규격 확인 (학습 없음)
_csv, _sub = T.subset(df, ["mrl", "dmd"], "train", limit=64)
_ds, _n = T.dataset_of(_csv, "train", SIZE, GRAY, False, 32, False)
xb, yb = next(iter(_ds))

print("배치 x :", tuple(xb.shape), xb.dtype.name,
      f"| 값 범위 {float(xb.numpy().min()):.1f} ~ {float(xb.numpy().max()):.1f}")
print("배치 y :", tuple(yb.shape), "| one-hot 합", float(yb.numpy().sum(axis=1)[0]))
print()
print("기대값 : x = (batch, 128, 128, 1), 0~255 float32  "
      "(정규화는 모델 첫 레이어 Rescaling(1/255) 이 담당)")
print("         y = (batch, 2) one-hot, class0 = Closed")

_m = T.build_eye_model(SIZE, 1 if GRAY else 3)
print(f"\n모델 파라미터 : {_m.count_params():,}")
print("모델 입력     :", _m.input_shape, "| 출력 :", _m.output_shape)

배치 x : (32, 128, 128, 1) float32 | 값 범위 0.0 ~ 255.0
배치 y : (32, 2) | one-hot 합 1.0

기대값 : x = (batch, 128, 128, 1), 0~255 float32  (정규화는 모델 첫 레이어 Rescaling(1/255) 이 담당)
         y = (batch, 2) one-hot, class0 = Closed

모델 파라미터 : 822,018
모델 입력     : (None, 128, 128, 1) | 출력 : (None, 2)


### 관찰 결과

- 입력 텐서와 라벨 규격이 기대값과 일치하는지 위 출력으로 확인한다.
- 값 범위가 0~1 로 나오면 이중 정규화다. 모델 안에 `Rescaling(1/255)` 이 있으므로 0~255 여야 한다.

### 목적

- 소규모 표본 · 1 epoch 로 학습 → 임계값 선택 → 평가 → 저장 전 구간을 통과시켜 오류를 미리 잡는다.
- 성능을 보는 셀이 아니다. 지표 값은 무의미하다.

In [3]:
# [셀 3] 스모크 학습 — 파이프라인 오류 확인 전용 (성능 해석 금지)
# limit 을 주면 산출물 이름에 __smoke 가 붙어 본 실험 결과를 덮어쓰지 않는다.
smoke = T.run_experiment(["mrl", "dmd"], eval_source="dmd", size=SIZE, gray=GRAY,
                         epochs=1, batch=32, limit=600, verbose=2)
print("\n스모크 통과. 아래 본 실험으로 진행한다.")

TAG  mrl+dmd__eval-dmd__gray128__smoke
train 600 (Closed 49.5%) | val 600 | test 600 (Closed 11.3%)


c:\Users\jiwoo\.conda\envs\drowsiness\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


19/19 - 8s - 396ms/step - accuracy: 0.5033 - loss: 0.6931 - val_accuracy: 0.5183 - val_loss: 0.6890

===== TEST =====
  [dmd test @val-thr] n=573  thr=0.46  Closed-Recall=0.7077  Precision=0.1281  F1=0.2170  Acc=0.4206
          실제Closed -> Closed 46 / Open 19   |   실제Open -> Closed 313 / Open 195
  [dmd test @0.50] n=573  thr=0.50  Closed-Recall=0.0000  Precision=0.0000  F1=0.0000  Acc=0.8866
          실제Closed -> Closed 0 / Open 65   |   실제Open -> Closed 0 / Open 508
  [mrl test @val-thr (대조군)] n=600  thr=0.46  Closed-Recall=0.6678  Precision=0.4582  F1=0.5435  Acc=0.4317
          실제Closed -> Closed 203 / Open 101   |   실제Open -> Closed 240 / Open 56

  -- 안경 조건별 (표본 적음, 참고용) --
  [glasses=0] n=431  thr=0.46  Closed-Recall=0.7826  Precision=0.0667  F1=0.1229  Acc=0.4037
          실제Closed -> Closed 18 / Open 5   |   실제Open -> Closed 252 / Open 156
  [glasses=1] n=142  thr=0.46  Closed-Recall=0.6667  Precision=0.3146  F1=0.4275  Acc=0.4718
          실제Closed -> Closed 28 / Open 14   

### 관찰 결과

- 학습 → val 임계값 선택 → test 평가 → 대조군 평가 → JSON 저장이 예외 없이 끝났다.
- 이 셀의 지표는 표본 600장 · 1 epoch 결과이므로 해석하지 않는다.

## PART B — 본 실험 (128 gray)

### 목적

- 네 조건을 같은 test(DMD hold-out 4명)에서 비교한다.
- 각 셀은 학습을 포함해 무겁다. 개별 실행이 가능하도록 결과를 `RESULTS` 에 누적한다.

In [4]:
# [셀 4] A) MRL only — 도메인 갭 하한선
RESULTS["A_mrl"] = T.run_experiment(["mrl"], eval_source="dmd",
                                    size=SIZE, gray=GRAY, epochs=EPOCHS, verbose=2)

TAG  mrl__eval-dmd__gray128
train 60,264 (Closed 49.2%) | val 12,479 | test 7,610 (Closed 11.3%)
Epoch 1/20


c:\Users\jiwoo\.conda\envs\drowsiness\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


942/942 - 263s - 279ms/step - accuracy: 0.9270 - loss: 0.1908 - val_accuracy: 0.8387 - val_loss: 0.4557
Epoch 2/20
942/942 - 142s - 150ms/step - accuracy: 0.9557 - loss: 0.1248 - val_accuracy: 0.8037 - val_loss: 0.5142
Epoch 3/20
942/942 - 130s - 138ms/step - accuracy: 0.9607 - loss: 0.1128 - val_accuracy: 0.8797 - val_loss: 0.3738
Epoch 4/20
942/942 - 127s - 135ms/step - accuracy: 0.9679 - loss: 0.0895 - val_accuracy: 0.8927 - val_loss: 0.3225
Epoch 5/20
942/942 - 130s - 138ms/step - accuracy: 0.9722 - loss: 0.0764 - val_accuracy: 0.8823 - val_loss: 0.3649
Epoch 6/20
942/942 - 136s - 144ms/step - accuracy: 0.9790 - loss: 0.0592 - val_accuracy: 0.9155 - val_loss: 0.2620
Epoch 7/20
942/942 - 136s - 144ms/step - accuracy: 0.9814 - loss: 0.0538 - val_accuracy: 0.9167 - val_loss: 0.2623
Epoch 8/20
942/942 - 134s - 142ms/step - accuracy: 0.9823 - loss: 0.0498 - val_accuracy: 0.9217 - val_loss: 0.2650
Epoch 9/20
942/942 - 152s - 162ms/step - accuracy: 0.9850 - loss: 0.0435 - val_accuracy: 0.

In [5]:
# [셀 5] B) DMD only — in-domain 상한선
RESULTS["B_dmd"] = T.run_experiment(["dmd"], eval_source="dmd",
                                    size=SIZE, gray=GRAY, epochs=EPOCHS, verbose=2)

TAG  dmd__eval-dmd__gray128
train 10,956 (Closed 51.4%) | val 3,338 | test 7,610 (Closed 11.3%)
Epoch 1/20


c:\Users\jiwoo\.conda\envs\drowsiness\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


172/172 - 52s - 301ms/step - accuracy: 0.7986 - loss: 0.4124 - val_accuracy: 0.7564 - val_loss: 0.8122
Epoch 2/20
172/172 - 26s - 149ms/step - accuracy: 0.9205 - loss: 0.2152 - val_accuracy: 0.8179 - val_loss: 0.4454
Epoch 3/20
172/172 - 26s - 151ms/step - accuracy: 0.9411 - loss: 0.1682 - val_accuracy: 0.8271 - val_loss: 0.5315
Epoch 4/20
172/172 - 26s - 150ms/step - accuracy: 0.9493 - loss: 0.1389 - val_accuracy: 0.8553 - val_loss: 0.4006
Epoch 5/20
172/172 - 25s - 148ms/step - accuracy: 0.9610 - loss: 0.1105 - val_accuracy: 0.8652 - val_loss: 0.4472
Epoch 6/20
172/172 - 26s - 149ms/step - accuracy: 0.9659 - loss: 0.1023 - val_accuracy: 0.9038 - val_loss: 0.2190
Epoch 7/20
172/172 - 25s - 148ms/step - accuracy: 0.9714 - loss: 0.0823 - val_accuracy: 0.9383 - val_loss: 0.1626
Epoch 8/20
172/172 - 25s - 148ms/step - accuracy: 0.9756 - loss: 0.0716 - val_accuracy: 0.9416 - val_loss: 0.1601
Epoch 9/20
172/172 - 26s - 148ms/step - accuracy: 0.9767 - loss: 0.0665 - val_accuracy: 0.9431 - va

In [6]:
# [셀 6] C) MRL + DMD 합본
RESULTS["C_mrl+dmd"] = T.run_experiment(["mrl", "dmd"], eval_source="dmd",
                                        size=SIZE, gray=GRAY, epochs=EPOCHS, verbose=2)

TAG  mrl+dmd__eval-dmd__gray128
train 71,220 (Closed 49.6%) | val 15,817 | test 7,610 (Closed 11.3%)
Epoch 1/20


c:\Users\jiwoo\.conda\envs\drowsiness\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


1113/1113 - 149s - 134ms/step - accuracy: 0.9208 - loss: 0.2070 - val_accuracy: 0.6622 - val_loss: 0.7923
Epoch 2/20
1113/1113 - 148s - 133ms/step - accuracy: 0.9459 - loss: 0.1584 - val_accuracy: 0.7149 - val_loss: 0.6600
Epoch 3/20
1113/1113 - 147s - 132ms/step - accuracy: 0.9592 - loss: 0.1240 - val_accuracy: 0.8093 - val_loss: 0.4644
Epoch 4/20
1113/1113 - 150s - 135ms/step - accuracy: 0.9683 - loss: 0.0910 - val_accuracy: 0.8613 - val_loss: 0.3611
Epoch 5/20
1113/1113 - 151s - 135ms/step - accuracy: 0.9759 - loss: 0.0688 - val_accuracy: 0.8780 - val_loss: 0.3015
Epoch 6/20
1113/1113 - 148s - 133ms/step - accuracy: 0.9798 - loss: 0.0584 - val_accuracy: 0.8747 - val_loss: 0.3368
Epoch 7/20
1113/1113 - 147s - 132ms/step - accuracy: 0.9822 - loss: 0.0513 - val_accuracy: 0.8900 - val_loss: 0.2931
Epoch 8/20
1113/1113 - 148s - 133ms/step - accuracy: 0.9837 - loss: 0.0489 - val_accuracy: 0.8837 - val_loss: 0.3481
Epoch 9/20
1113/1113 - 148s - 133ms/step - accuracy: 0.9850 - loss: 0.0448 

In [7]:
# [셀 7] C\') MRL 사전학습 -> DMD 미세조정  (셀 4 를 먼저 실행해야 한다)
_a = RESULTS.get("A_mrl")
_a_model = (config.PROJECT_ROOT / _a["model"]) if _a else (
    T.ARTIFACT_DIR / f"eye_mrl__eval-dmd__{'gray' if GRAY else 'rgb'}{SIZE}.keras")

if not Path(_a_model).exists():
    raise FileNotFoundError(f"A 조건 모델이 없습니다: {config._rel(Path(_a_model))}\n"
                            "  셀 4 를 먼저 실행하세요.")

# lr 을 1e-4 로 낮춘다. 기본 lr 로 미세조정하면 사전학습 가중치가 초기에 지워진다.
RESULTS["C'_ft"] = T.run_experiment(["dmd"], eval_source="dmd", pretrained=str(_a_model),
                                    lr=1e-4, size=SIZE, gray=GRAY, epochs=15, verbose=2)

TAG  dmd__eval-dmd__gray128__ft
train 10,956 (Closed 51.4%) | val 3,338 | test 7,610 (Closed 11.3%)
사전학습 로드: eye_mrl__eval-dmd__gray128.keras
Epoch 1/15


c:\Users\jiwoo\.conda\envs\drowsiness\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


172/172 - 25s - 143ms/step - accuracy: 0.9113 - loss: 0.2815 - val_accuracy: 0.9170 - val_loss: 0.2161
Epoch 2/15
172/172 - 22s - 130ms/step - accuracy: 0.9399 - loss: 0.1857 - val_accuracy: 0.9218 - val_loss: 0.1921
Epoch 3/15
172/172 - 23s - 132ms/step - accuracy: 0.9506 - loss: 0.1484 - val_accuracy: 0.9299 - val_loss: 0.1726
Epoch 4/15
172/172 - 23s - 133ms/step - accuracy: 0.9611 - loss: 0.1189 - val_accuracy: 0.9422 - val_loss: 0.1501
Epoch 5/15
172/172 - 23s - 133ms/step - accuracy: 0.9644 - loss: 0.1052 - val_accuracy: 0.9455 - val_loss: 0.1419
Epoch 6/15
172/172 - 23s - 134ms/step - accuracy: 0.9696 - loss: 0.0928 - val_accuracy: 0.9479 - val_loss: 0.1334
Epoch 7/15
172/172 - 23s - 133ms/step - accuracy: 0.9706 - loss: 0.0871 - val_accuracy: 0.9506 - val_loss: 0.1280
Epoch 8/15
172/172 - 23s - 134ms/step - accuracy: 0.9717 - loss: 0.0832 - val_accuracy: 0.9500 - val_loss: 0.1260
Epoch 9/15
172/172 - 23s - 134ms/step - accuracy: 0.9739 - loss: 0.0772 - val_accuracy: 0.9560 - va

In [9]:
# [셀 8] 조건 비교
# 결과는 artifacts 의 metrics JSON 에서 읽는다. 셀 4~7 을 따로 돌렸거나 커널을 다시
# 시작해도 동작한다(run_experiment 가 매번 JSON 을 쓰므로 디스크가 정본이다).
ALL = T.load_all_metrics()          # __smoke 는 제외된다
print(f"불러온 결과 {len(ALL)}건")
for _t in sorted(ALL):
    print("   ", _t)
print()

_main = {k: v for k, v in ALL.items() if not T.is_appendix(v)}
_appx = {k: v for k, v in ALL.items() if T.is_appendix(v)}

cmp = T.compare(_main)
if cmp.empty:
    raise RuntimeError("본 실험(128 gray) 결과가 없습니다. 셀 4~7 을 먼저 실행하세요.")

cmp.to_csv(COMPARISON_CSV, index=False)
print("저장 :", config._rel(COMPARISON_CSV))
print()
print("=== 본 실험 (128 gray) ===")
print(cmp.to_string(index=False))

if _appx:
    print("\n=== 부록 (입력 규격이 달라 본 실험과 대등 비교 아님) ===")
    print(T.compare(_appx).to_string(index=False))

print()
print("closed_recall       : DMD hold-out test, val 에서 고른 임계값 적용")
print("other_closed_recall : MRL test 9명 (대조군). DMD 를 올리려다 MRL 을 깎았는지 확인")

불러온 결과 5건
    dmd__eval-dmd__gray128
    dmd__eval-dmd__gray128__ft
    mrl+dmd__eval-dmd__gray128
    mrl+dmd__eval-dmd__rgb256
    mrl__eval-dmd__gray128

저장 : outputs\eye_dataset\step12_comparison.csv

=== 본 실험 (128 gray) ===
          cond                        tag  thr  closed_recall  closed_precision  closed_f1  accuracy  other_closed_recall  n_test_frames
     C MRL+DMD mrl+dmd__eval-dmd__gray128 0.93         0.9138            0.7871     0.8457    0.9624               0.8624           3805
C' MRL->DMD ft dmd__eval-dmd__gray128__ft 0.95         0.8811            0.8060     0.8419    0.9627               0.9195           3805
    A MRL only     mrl__eval-dmd__gray128 0.98         0.8531            0.3166     0.4618    0.7758               0.7284           3805
    B DMD only     dmd__eval-dmd__gray128 0.93         0.5967            0.9552     0.7346    0.9514               0.8709           3805

=== 부록 (입력 규격이 달라 본 실험과 대등 비교 아님) ===
             cond                       tag  th

| 조건               |  thr |     Recall |  Precision |         F1 |    MRL 대조군 |             오경보 |
| ---------------- | ---: | ---------: | ---------: | ---------: | ---------: | --------------: |
| **C (MRL+DMD)**  | 0.93 | **91.38%** | **78.71%** | **84.57%** |     86.24% |      106 (3.1%) |
| **C' (미세조정)**    | 0.95 |     88.11% | **80.60%** |     84.19% | **91.95%** |       91 (2.7%) |
| **A (MRL only)** | 0.98 |     85.31% |     31.66% |     46.18% |     72.84% | **790 (23.4%)** |
| **B (DMD only)** | 0.93 |     59.67% | **95.52%** |     73.46% |     87.09% |       12 (0.4%) |


### 관찰 결과

- **C(MRL+DMD)가 가장 균형적인 성능**을 보이며, Closed-Recall과 F1이 가장 높다.
- **C'는 C와 비슷한 성능**을 보이지만, Precision과 MRL 대조군 성능은 더 높고 Closed-Recall은 낮다.
- **A(MRL only)는 오경보가 많아** 실제 Open을 Closed로 잘못 판단하는 문제가 크다.
- **B(DMD only)는 Precision은 높지만 Closed-Recall이 낮아**, 실제 감은 눈을 많이 놓친다.
- 임계값 재선택 후 **A만 threshold가 0.95에서 0.98로 변경**되었으며, 나머지 조건은 기존 threshold가 유지되었다.

## PART C — 부록 · 연결 점검

### 목적

- 256 RGB 규격에서도 같은 결론이 나오는지 부록으로 1건 확인한다.
- 학습한 모델을 STEP03(YuNet 실시간 추론)에 넣을 수 있는지 **규격만** 점검한다. 교체 여부는 STEP13 결과까지 보고 정한다.

### 결정 박스 6 보완 — 임계값 그리드 상한

셀 8에서 A와 C'가 선택한 threshold 0.95는 당시 탐색 범위의 **최댓값**이었다. 따라서 더 높은 threshold에서 Recall 조건을 유지하면서 Precision이 추가로 개선될 가능성을 확인할 필요가 있었다.

이에 threshold grid를 `0.01 ~ 0.99999`로 확장했다. 0.99 이상은 로그 간격으로 탐색했으며, **재학습 없이** 저장된 모델을 불러와 threshold 선택과 test 평가만 다시 수행했다.

재평가 결과:

- **A:** 0.95 → **0.98**
- **B:** 0.93 → **0.93**
- **C:** 0.93 → **0.93**
- **C':** 0.95 → **0.95**
- **256 RGB 부록:** 0.91 → **0.91**

따라서 threshold 상한 문제는 A에서 실제로 영향을 주었다.

그러나 A의 threshold를 0.98까지 높였음에도 Precision은 기존 **0.3021에서 0.3166으로 1.5%p만 증가**했고, 실제 Open 3,376프레임 중 **790건(23.4%)** 을 여전히 Closed로 오판했다.

따라서 A의 낮은 Precision은 단순히 threshold 탐색 범위가 좁았기 때문에 발생한 결과라고 보기 어렵다.

두 셀 모두 메모리 변수 `RESULTS`가 아니라 `artifacts`의 metrics JSON을 읽으므로, 커널 재시작 후에도 저장된 결과를 기준으로 재평가할 수 있다.

In [8]:
# [셀 8b] 임계값 재선택 — 재학습 없음 (저장된 모델을 불러와 평가만 다시 한다)
# 이 셀도 디스크만 보므로 커널을 다시 시작해도 단독 실행된다.
TAGS = sorted(T.load_all_metrics())
print("재평가 대상", len(TAGS), "건\n")

RE = {}
for _tag in TAGS:
    RE[_tag] = T.reevaluate(_tag)
    print("-" * 70)

_re_main = {k: v for k, v in RE.items() if not T.is_appendix(v)}
_re_appx = {k: v for k, v in RE.items() if T.is_appendix(v)}

cmp2 = T.compare(_re_main)
cmp2.to_csv(COMPARISON_CSV, index=False)
print("갱신 저장 :", config._rel(COMPARISON_CSV))
print()
print("=== 본 실험 (128 gray) — 넓힌 임계값 그리드 ===")
print(cmp2.to_string(index=False))

if _re_appx:
    print("\n=== 부록 ===")
    print(T.compare(_re_appx).to_string(index=False))

재평가 대상 5 건

TAG  dmd__eval-dmd__gray128  (재평가 — 학습 없음)
이전 임계값 0.93 -> 새 그리드로 재선택

===== TEST =====
  [dmd test @val-thr] n=3,805  thr=0.93  Closed-Recall=0.5967  Precision=0.9552  F1=0.7346  Acc=0.9514
          실제Closed -> Closed 256 / Open 173   |   실제Open -> Closed 12 / Open 3364
  [dmd test @0.50] n=3,805  thr=0.50  Closed-Recall=0.8205  Precision=0.8502  F1=0.8351  Acc=0.9635
          실제Closed -> Closed 352 / Open 77   |   실제Open -> Closed 62 / Open 3314
  [mrl test @val-thr (대조군)] n=12,155  thr=0.93  Closed-Recall=0.8709  Precision=0.7538  F1=0.8081  Acc=0.7908
          실제Closed -> Closed 5355 / Open 794   |   실제Open -> Closed 1749 / Open 4257

  -- 안경 조건별 (표본 적음, 참고용) --
  [glasses=0] n=2,894  thr=0.93  Closed-Recall=0.7895  Precision=0.9434  F1=0.8596  Acc=0.9831
          실제Closed -> Closed 150 / Open 40   |   실제Open -> Closed 9 / Open 2695
  [glasses=1] n=911  thr=0.93  Closed-Recall=0.4435  Precision=0.9725  F1=0.6092  Acc=0.8507
          실제Closed -> Closed 106 / Open 133

In [9]:
# [셀 9] 부록 — 256 RGB 조건 (STEP02 의 eye_model.keras 와 같은 입력 규격)
# 본 실험이 아니다. 결정 박스 8의 판단이 규격을 바꿔도 유지되는지 확인하는 용도.
RESULTS["appendix_rgb256"] = T.run_experiment(
    ["mrl", "dmd"], eval_source="dmd", size=256, gray=False,
    epochs=10, batch=32, verbose=2)

TAG  mrl+dmd__eval-dmd__rgb256
train 71,220 (Closed 49.6%) | val 15,817 | test 7,610 (Closed 11.3%)
Epoch 1/10


c:\Users\jiwoo\.conda\envs\drowsiness\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


2226/2226 - 614s - 276ms/step - accuracy: 0.9333 - loss: 0.1899 - val_accuracy: 0.7874 - val_loss: 0.4817
Epoch 2/10
2226/2226 - 613s - 275ms/step - accuracy: 0.9537 - loss: 0.1272 - val_accuracy: 0.8588 - val_loss: 0.3617
Epoch 3/10
2226/2226 - 611s - 274ms/step - accuracy: 0.9599 - loss: 0.1140 - val_accuracy: 0.8908 - val_loss: 0.3149
Epoch 4/10
2226/2226 - 606s - 272ms/step - accuracy: 0.9666 - loss: 0.0924 - val_accuracy: 0.8641 - val_loss: 0.4227
Epoch 5/10
2226/2226 - 594s - 267ms/step - accuracy: 0.9691 - loss: 0.0856 - val_accuracy: 0.8490 - val_loss: 0.4506
Epoch 6/10
2226/2226 - 591s - 266ms/step - accuracy: 0.9742 - loss: 0.0731 - val_accuracy: 0.8963 - val_loss: 0.3736
Epoch 7/10
2226/2226 - 602s - 270ms/step - accuracy: 0.9758 - loss: 0.0698 - val_accuracy: 0.8879 - val_loss: 0.3409
Epoch 8/10
2226/2226 - 598s - 269ms/step - accuracy: 0.9785 - loss: 0.0616 - val_accuracy: 0.9093 - val_loss: 0.2880
Epoch 9/10
2226/2226 - 591s - 266ms/step - accuracy: 0.9786 - loss: 0.0609 

In [10]:
# [셀 10] STEP03 연결 가능성 — 규격 비교만 한다 (교체하지 않는다)
import tensorflow as tf

_existing = T.ARTIFACT_DIR / "eye_model.keras"       # STEP02 산출물, STEP03 이 로드하는 모델
rows = []

if _existing.exists():
    _old = tf.keras.models.load_model(_existing)
    rows.append(dict(model="eye_model.keras (STEP02)", input=str(_old.input_shape),
                     output=str(_old.output_shape), params=_old.count_params()))

for k, r in RESULTS.items():
    if "smoke" in r["tag"]:
        continue                                     # 스모크 산출물은 규격 비교 대상이 아니다
    p = config.PROJECT_ROOT / r["model"]
    if p.exists():
        m = tf.keras.models.load_model(p)
        rows.append(dict(model=f"{k} ({r['tag']})", input=str(m.input_shape),
                         output=str(m.output_shape), params=m.count_params()))

spec = pd.DataFrame(rows)
print(spec.to_string(index=False))
print()
print("STEP03 은 (None, 256, 256, 3) 입력에 0~255 BGR->RGB crop 을 넣고 class0=Closed 를 읽는다.")
print("입력 shape 가 다른 모델은 STEP03 코드를 고치지 않고는 로드해도 동작하지 않는다.")
print("STEP03 은 수정 금지 대상이므로, 교체는 규격이 일치하는 모델로만 가능하다.")

                                      model               input    output  params
                   eye_model.keras (STEP02) (None, 256, 256, 3) (None, 2) 3706178
             A_mrl (mrl__eval-dmd__gray128) (None, 128, 128, 1) (None, 2)  822018
             B_dmd (dmd__eval-dmd__gray128) (None, 128, 128, 1) (None, 2)  822018
     C_mrl+dmd (mrl+dmd__eval-dmd__gray128) (None, 128, 128, 1) (None, 2)  822018
         C'_ft (dmd__eval-dmd__gray128__ft) (None, 128, 128, 1) (None, 2)  822018
appendix_rgb256 (mrl+dmd__eval-dmd__rgb256) (None, 256, 256, 3) (None, 2) 3706178

STEP03 은 (None, 256, 256, 3) 입력에 0~255 BGR->RGB crop 을 넣고 class0=Closed 를 읽는다.
입력 shape 가 다른 모델은 STEP03 코드를 고치지 않고는 로드해도 동작하지 않는다.
STEP03 은 수정 금지 대상이므로, 교체는 규격이 일치하는 모델로만 가능하다.


### 관찰 결과

- **C(MRL+DMD)가 가장 균형적인 성능**을 보였다. Closed-Recall과 F1이 가장 높다.
- **A(MRL only)는 오경보가 많았다.** 실제 Open을 Closed로 잘못 판단하는 비율이 높게 나타났다.
- **B(DMD only)는 Precision은 높지만 Closed-Recall이 낮았다.** 실제 감은 눈을 많이 놓쳤다.
- **C와 C'는 전반적으로 비슷한 성능**을 보였으며, C'는 MRL 대조군에서 더 높은 성능을 보였다.
- 임계값 재선택 후 **A만 threshold가 0.95에서 0.98로 변경**되었고, 나머지 조건은 그대로 유지됐다.


## 해석

- 동일한 DMD hold-out test set에서 비교했을 때, **C(MRL+DMD)가 네 조건 중 가장 높은 Closed-Recall과 F1을 기록했다.** 따라서 본 실험에서는 C를 후속 단계의 기본 모델로 선택한다.
- A(MRL only)는 DMD 환경에서 Closed-Recall 자체는 85.31%였지만 Precision이 31.66%로 낮았고, MRL Open의 23.4%를 Closed로 판정했다. 이는 **MRL에서 학습한 모델을 DMD 환경에 직접 적용할 때 오경보가 증가할 수 있음**을 보여준다.
- B(DMD only)는 Precision은 높았지만 Closed-Recall이 59.67%로 낮아, **DMD 표본만으로 학습한 경우 감은 눈을 충분히 포착하지 못했을 가능성**이 있다.
- C와 C'는 F1이 각각 84.57%, 84.19%로 유사했다. 따라서 현재 실험에서는 **MRL+DMD 합본과 MRL→DMD 미세조정 사이에 큰 F1 차이는 확인되지 않았다.**
- A의 threshold를 0.98까지 확장한 뒤에도 Open→Closed 오판 비율이 23.4%로 남아, **A의 낮은 Precision이 단순히 threshold 탐색 범위의 문제만으로 설명되지는 않는다.**
- 다만 이 결과는 단일 seed와 하나의 고정 DMD hold-out split에서 얻어진 것이므로, **MRL+DMD가 일반적으로 항상 우수하다고 일반화하기보다는 본 실험 조건에서 가장 유리한 결과를 보였다고 해석한다.**


## 한계

- 각 조건을 **단일 seed로 1회 학습**했다.
- 조건별 **validation domain과 threshold 선택 기준이 동일하지 않다.**
- DMD는 **정면 고정 카메라·Car Stopped 조건**이므로 실제 주행 환경에 그대로 일반화하기 어렵다.
- 256 RGB는 학습 조건이 달라 **128 gray와 직접 비교할 수 없다.**



## STEP12 요약

### Takeaway

- **C(MRL+DMD)를 후속 모델로 선택한다.**
- MRL only는 DMD 환경에서 오경보가 많았고, DMD only는 감은 눈을 많이 놓쳤다.
- 따라서 현재 데이터에서는 **서로 다른 도메인의 데이터를 함께 학습하는 방식이 더 안정적인 결과**를 보였다.
- **STEP13은 C 모델(threshold 0.93)을 기준으로 진행한다.**